# SILENCE ENGINE on LinuxONE — run it from this notebook

*Detect what should have happened, but didn't.* All data here is **synthetic** (fictional sites and events).

**Before you start:** upload `silence-engine-1.0.0.tar.gz` (the release built on the Mac) into the **same folder as this notebook** using Jupyter's *Upload* button. Do not paste source files by hand.

Run the cells from top to bottom:

* **Part A** runs the detection engine and the evaluation **inside this kernel**, using only the Python standard library. No installs or internet needed, and it proves the engine runs on this machine's processor.
* **Part B** installs the web backend in its own virtual environment, runs the test suite, starts the dashboard in the background, and shows how to open it.
* **Part C** (optional) trains a small scikit-learn model on the synthetic *training* period and benchmarks it against the rule-based triage on the *test* period, in this kernel. It needs `scikit-learn` and `numpy`, which the LinuxONE ML image ships (PyPI has no s390x wheels for them, so do not `pip install` them here). Without them Part C says so and everything else still works.


In [ ]:
# 1. Where are we running?  (read-only)
import os, platform, sys
from pathlib import Path

print("Processor architecture:", platform.machine(), "(s390x = IBM Z / LinuxONE)" if platform.machine() == "s390x" else "")
print("Byte order:            ", sys.byteorder)
print("Python (this kernel):  ", platform.python_version(), sys.executable)
if Path("/etc/os-release").exists():
    print("OS:                    ", next((l.split("=", 1)[1].strip().strip('"') for l in Path("/etc/os-release").read_text().splitlines()
                                            if l.startswith("PRETTY_NAME=")), "?"))
if Path("/proc/sysinfo").exists():
    for line in Path("/proc/sysinfo").read_text(errors="replace").splitlines():
        if line.split(":")[0].strip() in ("Manufacturer", "Type", "Model"):
            print("/proc/sysinfo:         ", " ".join(line.split()))
print("User:                  ", "root" if hasattr(os, "geteuid") and os.geteuid() == 0 else os.environ.get("USER", "?"))
print("Working folder:        ", Path.cwd())
if sys.version_info < (3, 10):
    print("\nWARNING: this kernel is older than Python 3.10. Part A may still work; Part B will look for a newer python3.x.")

In [ ]:
# 2. Unpack the uploaded release (skipped if already unpacked, so your data is kept)
import tarfile

TARBALL = Path("silence-engine-1.0.0.tar.gz")
APP = Path("silence-engine-1.0.0").resolve()
PORT = int(os.environ.get("SILENCE_PORT", "8000"))   # web port used in Part B

if APP.exists():
    print("Already unpacked:", APP)
else:
    if not TARBALL.exists():
        raise FileNotFoundError(f"Upload {TARBALL} into {Path.cwd()} first (Jupyter file browser > Upload).")
    with tarfile.open(TARBALL) as tar:
        for m in tar.getmembers():   # refuse absolute paths and '..' (path traversal)
            if m.name.startswith("/") or ".." in Path(m.name).parts:
                raise ValueError(f"Unsafe path in archive: {m.name}")
        if hasattr(tarfile, "data_filter"):
            tar.extractall(Path.cwd(), filter="data")   # safe extraction on newer Pythons
        else:
            tar.extractall(Path.cwd())
    print("Unpacked to", APP)
print(sorted(p.name for p in APP.iterdir()))

## Part A — the engine in this kernel (standard library only)

In [ ]:
# 3. Generate the synthetic dataset and run detection + triage
import collections, time
sys.path.insert(0, str(APP / "backend"))
from silence_engine import data_generator, ingest, pipeline

t0 = time.time()
ds = data_generator.generate(seed=42)
obs = ingest.validate_records(ds.raw_records, source="synthetic").observations
result = pipeline.analyze(obs, ds.sites)
print(f"{len(obs):,} synthetic records, {len(result.streams)} service streams, analysed in {time.time() - t0:.2f}s on {platform.machine()}")
print("Dataset fingerprint:", ds.manifest["fingerprint_sha256"])
print("Cases by category:  ", dict(collections.Counter(c["category"] for c in result.cases)))
print("(shipped default thresholds; the dashboard in Part B uses thresholds tuned on the training period)")
print("\nTop open investigations (hypotheses to verify, not findings):")
for c in result.actionable_cases()[:6]:
    print(f"  [{c['priority']:>3}] {c['category_label']:<30} {c['site_name']} / {c['service_type']}  {c['onset']} → {c['end']}  feed {c['feed_status']}")

In [ ]:
# 4. Evaluate against held-out synthetic scenarios (about 15 s on a laptop; includes the delay replay)
from silence_engine import evaluation

t0 = time.time()
report = evaluation.evaluate(obs, ds.sites, ds.labels, include_replay=True)
rows = [("Event recall", "detection", "recall"), ("Alert precision", "detection", "precision"),
        ("Explanation accuracy", "explanation", "accuracy_given_detection"),
        ("Feed failures called access gaps", "explanation", "reporting_failures_called_access_gaps"),
        ("False alerts /100 stream-weeks", "detection", "false_alerts_per_100_stream_weeks")]
print(f"Test period (synthetic), evaluated in {time.time() - t0:.1f}s\n")
print(f"{'':36}{'Naive % drop':>14}{'Ablation':>12}{'ENGINE':>10}")
for label, part, key in rows:
    vals = [report["test"][m][part][key] for m in ("naive", "robust", "engine")]
    fmt = (lambda v: f"{v:.1f}") if "per_100" in key else (lambda v: "n/a" if v is None else f"{v:.0%}")
    print(f"{label:36}{fmt(vals[0]):>14}{fmt(vals[1]):>12}{fmt(vals[2]):>10}")
print("\nResults hash:", report["results_sha256"])
MAC = "e201d75415c8d1aaf34d6af6b85cb28aa788d6f51207acb1e24cac734532ea1d"
print("Same as the Mac reference run:", "YES — bit-for-bit reproducible" if report["results_sha256"] == MAC else "NO — please record this")

## Part B — the web dashboard on this server

This creates `silence-engine-1.0.0/.venv`, installs Flask + gunicorn from PyPI (needs internet; MarkupSafe compiles from source on s390x or falls back to pure Python), and generates the demo database. It takes a few minutes the first time.

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# 5. Install the backend into its own virtual environment
import subprocess

env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"   # Jupyter containers often run as root; the scripts refuse root otherwise
import importlib.util
if importlib.util.find_spec("sklearn") and sys.version_info >= (3, 10):
    # This kernel has the ML stack (as the LinuxONE ML image does): build the dashboard's venv from this same interpreter and
    # let it see those packages, so the dashboard's ML section works too. Without scikit-learn nothing changes.
    env["PYTHON"] = sys.executable
    env["SILENCE_VENV_SYSTEM_SITE"] = "1"
proc = subprocess.run(["bash", "scripts/linuxone/install.sh", "--skip-tests"], cwd=APP, env=env,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lines = [l for l in proc.stdout.splitlines() if not l.startswith(("Requirement already", "  Using cached", "Collecting", "Downloading"))]
print("\n".join(lines[-40:]))
print("\nInstall", "OK" if proc.returncode == 0 else f"FAILED (exit {proc.returncode}) — see docs/IBM_Z_DEPLOYMENT.md, Troubleshooting")

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# 6. Run the full automated test suite on this machine
proc = subprocess.run([str(APP / ".venv/bin/python"), "-m", "pytest", "-o", "addopts=", "-q", "--color=no", "-p", "no:cacheprovider"], cwd=APP / "backend",
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout[-2500:])
print("Tests", "PASSED" if proc.returncode == 0 else "FAILED — keep this output, it is a real portability finding")

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# 7. Start the dashboard in the background (keeps running if the kernel restarts)
import json, urllib.request

proc = subprocess.run(["bash", "scripts/linuxone/run.sh", "--background"], cwd=APP, env=env,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(proc.stdout.splitlines()[0] if proc.stdout else proc)
for _ in range(20):
    try:
        health = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=3))
        print("Backend health:", health["status"], "| architecture:", health["runtime"]["machine"], "| data as of", health["as_of"])
        break
    except Exception:
        time.sleep(1)
else:
    print("Not answering yet — check", APP / "data/server.log")

### 8. Open the dashboard

* **If this Jupyter has `jupyter-server-proxy`** (the next cell checks), open the link it prints **in a new browser tab**. The dashboard uses relative URLs, so it works under the `/proxy/PORT/` prefix.
* **Otherwise**, if this Jupyter runs on the VM you can SSH into, run this on your Mac and open <http://127.0.0.1:8080>:
  `ssh -i <your key> -N -L 8080:127.0.0.1:8000 linux1@<server-ip>`
* Do not open the port to the internet. Write actions (reviews, regenerate, import) are accepted only from the server itself, and the proxy and tunnel both arrive there.

In [ ]:
# 8. Is the dashboard running, and where do I open it?
import os, json, urllib.request, importlib.util
from IPython.display import HTML, display
PORT = int(os.environ.get("SILENCE_PORT", "8000"))

try:
    h = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=4))
    print(f"Server is UP on port {PORT}: status={h['status']}, architecture={h['runtime']['machine']}, data as of {h['as_of']}")
except Exception as exc:
    print(f"Server is NOT answering on port {PORT} ({exc.__class__.__name__}). Run cell 7 (Start the dashboard) first,")
    print("then check the log:  silence-engine-1.0.0/data/server.log")

has_proxy = importlib.util.find_spec("jupyter_server_proxy") is not None
print("jupyter-server-proxy:", "installed" if has_proxy else "NOT installed (use the SSH tunnel described above)")
display(HTML(f'''<p><a id="se-link" target="_blank" rel="noopener" href="#">Open SILENCE ENGINE in a new tab</a>
<br><span id="se-url" style="font-family:monospace"></span></p>
<script>
(function () {{
  var p = window.location.pathname;                       // e.g. /lab/tree/shared/x.ipynb  or  /user/me/lab/...
  var cut = p.search(/\\/(lab|notebooks|tree|doc|files)(\\/|$)/);
  var base = cut >= 0 ? p.slice(0, cut) : "";
  var url = window.location.origin + base + "/proxy/{PORT}/";
  var a = document.getElementById("se-link"); a.href = url;
  document.getElementById("se-url").textContent = url;
}})();
</script>'''))
print("If the link is blank, type this address by hand: http://<server-ip>:<jupyter-port>/proxy/%d/" % PORT)

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# 9. Reproducibility fingerprints from the installed backend (compare with the Mac)
print(subprocess.run([str(APP / ".venv/bin/python"), "backend/manage.py", "fingerprint"], cwd=APP,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

## Part C — optional machine-learning triage, benchmarked against the rules (in this kernel)

The detector decides *what* is flagged. Triage then decides *which kind of silence* an episode is: potential access gap, likely reporting problem, or uncertain. The rule-based triage does this with documented thresholds. Here a **random forest** (scikit-learn) learns the same decision from examples:

```
records → feed-health checks → baselines, z-score, CUSUM → flagged episodes → 19 evidence features ─┬→ RandomForest (300 trees, depth 4) → category   (optional)
                                                                                                  └→ rule-based triage                  → category   (the default)
```

* **Trained only on the synthetic training period** (records and labels dated up to its end). **Scored only on the test period**, by the same function and confidence intervals as the rules.
* The rule-based verdict is never an input to the model. Ground-truth labels never reach the detector or the pipeline.
* It is an honest benchmark, not a sales pitch: with ~40 training episodes the model is expected to land close to the rules. The dashboard keeps the rules as the default.
* PyPI has no s390x wheels for numpy/scikit-learn. Use the **LinuxONE ML image's kernel** (it ships scikit-learn, XGBoost, pandas, TensorFlow and PyTorch for s390x). Run Part A first (Part C reuses its data and tuned thresholds).

In [ ]:
# C1. Is the ML stack available in THIS kernel, and on what hardware? (read-only)
from silence_engine import ml_triage

status = ml_triage.availability()
print(f"Processor architecture: {platform.machine()}  ({sys.byteorder}-endian)  Python {platform.python_version()}  {sys.executable}")
print("Packages visible to this kernel (the ML ones are optional):")
for name, version in ml_triage.package_versions().items():
    print(f"  {name:<14}{version or '-'}")
print("\nML triage can run in this kernel:", status["available"])
if not status["available"]:
    print(status["reason"])

In [ ]:
# C2. Train on the TRAINING period, score on the TEST period, compare with the rule-based triage (about 10-60 s)
from silence_engine import ml_evaluation
from silence_engine.detection import DetectionParams

ml = None
if not ml_triage.availability()["available"]:
    print("Skipped: scikit-learn / numpy are not importable in this kernel (see the message above). Parts A and B are unaffected.")
elif "report" not in globals():
    print("Skipped: run Part A (cells 3 and 4) first; Part C reuses its data and the thresholds tuned on the training period.")
else:
    params = DetectionParams.from_dict(report["engine_params_full"])   # thresholds tuned on the training period by Part A
    t0 = time.time()
    ml = ml_evaluation.evaluate_ml(obs, ds.sites, ds.labels, params)
    print(f"Done in {time.time() - t0:.1f}s on {platform.machine()}\n")
    print(ml_evaluation.format_comparison(ml))

In [ ]:
# C3. Model architecture and the evidence it reads (what the forest is actually given)
if ml:
    print(f"{ml['model']['name']}  {ml['model']['spec']}")
    print(f"Trained on {ml['training']['episodes']} episodes {ml['training']['by_target']} from {ml['training']['window'][0]} to {ml['training']['window'][1]}; classes: {ml['model']['classes']}")
    print(ml["model"]["inputs_excluded"], "\n")
    for f in ml["model"]["features"]:
        print(f"  {f['name']:<24}{f['description']}")
    print("\nHonest read-out:", ml["conclusion"]["headline"])
    print("Production classifier:", ml["production_classifier"]["name"], "-", ml["production_classifier"]["reason"])

In [ ]:
# C4. Reproducibility: compare with the reference run on the Mac (arm64, little-endian)
MAC_RESULTS = "cbea084debb002b549a35ace56d3a1254215a6a723e6f28fdd8b1e193effb1d8"        # Python 3.11 + scikit-learn 1.5.1 and Python 3.13 + scikit-learn 1.7.2 gave this exact value
MAC_PREDICTIONS = "6949d3d632e4b8b5479b86465eeee289d7b2a5e6ddf63e019bf531937a1a68f9"     # hash of the per-episode predicted categories only
if ml:
    print("results_sha256     ", ml["results_sha256"])
    print("predictions_sha256 ", ml["predictions_sha256"])
    print("Same predictions as the Mac reference:", "YES" if ml["predictions_sha256"] == MAC_PREDICTIONS else "NO - record this together with the library versions above (a real portability finding)")
    print("Same full results hash as the Mac:     ", "YES" if ml["results_sha256"] == MAC_RESULTS else "NO - may differ with library versions; the predictions hash above is the stricter portability check")
    print("Ran on:", ml["runtime"]["machine"], f"({ml['runtime']['byteorder']}-endian)", ml["runtime"]["libraries"])

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# C5. (needs Part B) the dashboard's API serves the same comparison, computed by its own process
if ml:
    try:
        api = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/ml-triage", timeout=10))
        if api.get("available") and api.get("report"):
            print("Dashboard API results_sha256 matches this kernel:", api["report"]["results_sha256"] == ml["results_sha256"], "| open Evaluation > 'Optional: machine-learning triage' in the dashboard")
        else:
            print("The dashboard answered but its ML section is not available:", api.get("reason") or "not run yet")
    except Exception as exc:
        print("Dashboard not reachable (run Part B cells 5 and 7 first):", type(exc).__name__)

In [ ]:
# --- self-contained setup: this cell works even if earlier cells were skipped ---
import os, sys, time, json, subprocess, urllib.request
from pathlib import Path
PORT = int(os.environ.get("SILENCE_PORT", "8000"))
APP = Path("silence-engine-1.0.0").resolve()
env = dict(os.environ, SILENCE_PORT=str(PORT))
if hasattr(os, "geteuid") and os.geteuid() == 0:
    env["SILENCE_ALLOW_ROOT"] = "1"
if not APP.exists():
    raise FileNotFoundError("Run cell 2 first (it unpacks silence-engine-1.0.0.tar.gz next to this notebook).")
# --------------------------------------------------------------------------------
# 10. Stop the dashboard when you are done
print(subprocess.run(["bash", "scripts/linuxone/stop.sh"], cwd=APP, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)